In [13]:
import pandas as pd

# ============================================
# LOAD DATASET
# ============================================
df = pd.read_csv("../data/raw/sms_spam_indo.csv")
df["Kategori_NusaGuard"] = None

# ============================================
# 1. AMAN — semua ham dulu
# ============================================
df.loc[df["Kategori"] == "ham", "Kategori_NusaGuard"] = "Aman"

# ============================================
# 2. PHISHING/LINK BERBAHAYA — cek link DULUAN, sebelum promo resmi
#    (supaya link scam yg kebetulan mengandung nama brand tidak lolos)
# ============================================
sisa = df["Kategori_NusaGuard"].isna()
pola_phish = r"(https?://|www\.|bit\.ly|\.(com|co\.id|co\.vu|net|info|tk|ga|ml|blogspot|jimdo|weebly|webnode|webs)\b)"
mask_phish = sisa & df["Pesan"].str.contains(pola_phish, case=False, na=False, regex=True)
df.loc[mask_phish, "Kategori_NusaGuard"] = "Phishing/Link Berbahaya"

# ============================================
# 3. PENIPUAN ROMANSA — cek sebelum promo resmi
#    (supaya "yank isi pulsa" tidak kebaca sbg promo pulsa biasa)
# ============================================
sisa = df["Kategori_NusaGuard"].isna()
pola_romansa = r"\b(yank|sayang|ma[\",']|papa|mama)\b.*\b(pulsa|transfer|kirim)\b"
mask_romansa = sisa & df["Pesan"].str.contains(pola_romansa, case=False, na=False, regex=True)
df.loc[mask_romansa, "Kategori_NusaGuard"] = "Penipuan Romansa"

# ============================================
# 4. PENIPUAN INVESTASI
# ============================================
sisa = df["Kategori_NusaGuard"].isna()
pola_investasi = r"(pinjaman|kredit tanpa|\bkta\b|bunga [0-9]|investasi|togel|modal usaha|jaminan bpkb|pelunasan|cc\s*/\s*kta)"
mask_investasi = sisa & df["Pesan"].str.contains(pola_investasi, case=False, na=False, regex=True)
df.loc[mask_investasi, "Kategori_NusaGuard"] = "Penipuan Investasi"

# ============================================
# 5. PENIPUAN REKRUTMEN
# ============================================
sisa = df["Kategori_NusaGuard"].isna()
pola_rekrutmen = r"(lowongan|kirim cv|lamaran|ijazah|beasiswa|seminar.*dikti|jobfair)"
mask_rekrutmen = sisa & df["Pesan"].str.contains(pola_rekrutmen, case=False, na=False, regex=True)
df.loc[mask_rekrutmen, "Kategori_NusaGuard"] = "Penipuan Rekrutmen"

# ============================================
# 6. AMAN — baru sekarang cek promo resmi/legal, dari sisa yg belum kena scam manapun
# ============================================
sisa = df["Kategori_NusaGuard"].isna()
pola_resmi = r"tsel\.me|indosatooredoo\.com|mytsel|domino|kfc|grab\.co|panpizza|myxl|axisworld|tcash|bimatri\b"
mask_resmi = sisa & df["Pesan"].str.contains(pola_resmi, case=False, na=False, regex=True)
df.loc[mask_resmi, "Kategori_NusaGuard"] = "Aman"

sisa = df["Kategori_NusaGuard"].isna()
pola_promo_signal = r"paket|kuota|\bgb\b|\bmb\b|pulsa|bonus|diskon|disc\.|promo|s&k|info838|\*\d{2,3}(\*\d+)*#|\btsel\b|\baxis\b|\bxl\b|smartfren|\bthree\b|indosatooredoo"
pola_scam_signal = r"pin\s*(pemenang|anda)?|hadiah|menang|rekening|no\.?\s*rek|transfer|kode cek|survey rumah|jaminan bpkb|kirim uang|kirim pulsa"
mask_promo2 = sisa & df["Pesan"].str.contains(pola_promo_signal, case=False, na=False, regex=True) & ~df["Pesan"].str.contains(pola_scam_signal, case=False, na=False, regex=True)
df.loc[mask_promo2, "Kategori_NusaGuard"] = "Aman"

# ============================================
# 7. FALLBACK — sisanya Social Engineering
# ============================================
sisa = df["Kategori_NusaGuard"].isna()
df.loc[sisa, "Kategori_NusaGuard"] = "Social Engineering"

# ============================================
# HASIL & VALIDASI
# ============================================
print("=== Distribusi Kategori_NusaGuard ===")
print(df["Kategori_NusaGuard"].value_counts())

print("\n=== Sisa Social Engineering (sample 10) ===")
print(df[df["Kategori_NusaGuard"]=="Social Engineering"]["Pesan"].sample(10, random_state=1).tolist())

print("\n=== VALIDASI: spam yang jadi 'Aman' — cek ada yg lolos scam? ===")
cek_aman_dari_spam = df[(df["Kategori"]=="spam") & (df["Kategori_NusaGuard"]=="Aman")]
print("Jumlah spam yang jadi Aman:", len(cek_aman_dari_spam))
print(cek_aman_dari_spam["Pesan"].sample(15, random_state=2).tolist())

# ============================================
# SIMPAN
# ============================================
df.to_csv("../data/processed/sms_relabeled_draft.csv", index=False)
print("\nTersimpan. Total baris:", len(df))

=== Distribusi Kategori_NusaGuard ===
Kategori_NusaGuard
Aman                       766
Phishing/Link Berbahaya    235
Social Engineering         109
Penipuan Investasi          23
Penipuan Romansa             7
Penipuan Rekrutmen           3
Name: count, dtype: int64

=== Sisa Social Engineering (sample 10) ===
['Kejutan POIN, hadian Rp 15 jt resmi diberikan untuk pemilik nomor 0812330xxxx. PIN-x7af2547. Untuk Info Kantor Pusat TELKOMSEL 021-23867047, 021-23867048.', 'Hayu dateng, mumpung ada ustad Fatih karim,beliau yg mengislamkan ustad Felix siaw', "Ass, Sy Randy. Mengenai mobil yg sdh sy liat kondisi'y , kebetulan kami berminat. Mohon hub Bapa sy utk dibicarakan hrg netnya; Dr.H.DARMAWAN 0812xxxxxxx", 'Nanti kalau mau di bayar/di transfer harap hubungi dulu atasan saya Bpk ARDIANSA no tlp; 085797136879 soalnya ada perubahan pembayaran trima kasih.', 'Uangnya dikirim aja ke Rekening ini, BankBNI a/n SAFITRI \xa0No.Rekening 0282566132', 'Maaf‟ Cuma ingatkan soal kontrakan kalau mau 

C:\Users\ASUS\AppData\Local\Temp\ipykernel_17588\2852125011.py:20: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_phish = sisa & df["Pesan"].str.contains(pola_phish, case=False, na=False, regex=True)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_17588\2852125011.py:29: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_romansa = sisa & df["Pesan"].str.contains(pola_romansa, case=False, na=False, regex=True)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_17588\2852125011.py:37: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_investasi = sisa & df["Pesan"].str.contains(pola_investasi, case=False, na=False, regex=True)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_17588\2852125011.py:45: UserWarning: This pattern is interpreted a